In [1]:
from brian2 import *

import numpy as np
import os
from glob import glob
import cv2
import sys
sys.path.insert(0, r"/home/jake/Document/Spikes/spikes")
from network import *
from input import *
from projects import *
import tensorflow as tf
import tensorflow_datasets as tfds


/home/jake/Document/Spikes/.linuxvenv/lib/python3.12/site-packages/setuptools/_distutils/_msvccompiler.py:12: UserWarning: _get_vc_env is private; find an alternative (pypa/distutils#340)
  warnings.warn(
2025-06-20 15:01:23.635822: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750431683.656651   16502 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750431683.663033   16502 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750431683.680351   16502 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750431683.680373   16502 computation_placer.

In [3]:
def load_gabor_filters(filter_dir):
    """
    Load all Gabor filter .npy files from a directory into a list of arrays.
    
    Args:
        filter_dir (str): Path to directory containing filter .npy files
        
    Returns:
        list: List of numpy arrays, each representing a Gabor filter
    """
    # Ensure path exists
    if not os.path.exists(filter_dir):
        raise FileNotFoundError(f"Filter directory not found: {filter_dir}")
    
    # Get all .npy files in the directory
    filter_files = glob(os.path.join(filter_dir, "*.npy"))
    
    if not filter_files:
        print(f"No .npy files found in {filter_dir}")
        return []
    
    # Load each filter into a list
    filters = []
    for file_path in filter_files:
        try:
            filter_array = np.load(file_path)
            filters.append(filter_array)
            print(f"Loaded filter from {os.path.basename(file_path)}, shape: {filter_array.shape}")
        except Exception as e:
            print(f"Error loading {file_path}: {str(e)}")
    
    print(f"Loaded {len(filters)} Gabor filters")
    return filters

def upscale_mnist(images, target_size=128, method='bicubic'):
    """
    Upscale MNIST images from 28x28 to target_size x target_size
    
    Args:
        images: NumPy array with shape (n_images, 28, 28)
        target_size: Target size (default: 128)
        method: Upscaling method ('nearest', 'bilinear', 'bicubic', or 'lanczos')
        
    Returns:
        NumPy array with shape (n_images, target_size, target_size)
    """
    num_images = images.shape[0]
    upscaled = np.zeros((num_images, target_size, target_size))
    
    for i in range(num_images):
        # OpenCV resize
        if method == 'nearest':
            interpolation = cv2.INTER_NEAREST
        elif method == 'bilinear':
            interpolation = cv2.INTER_LINEAR
        elif method == 'bicubic':
            interpolation = cv2.INTER_CUBIC
        elif method == 'lanczos':
            interpolation = cv2.INTER_LANCZOS4
        else:
            raise ValueError(f"Unknown method: {method}")
        
        # OpenCV takes (width, height) instead of (height, width)
        upscaled[i] = cv2.resize(images[i], (target_size, target_size), interpolation=interpolation)
        
    return upscaled


In [ ]:
# batch_size = 50
# noiseAmount = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
# conv_dir = r"/home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/conv_mnist_batch/"
# neuron_input_dir = r"C:\Users\reidj\Dropbox\dphil\programming\spikes\projects\mnist_class\mnist_class_wip\data\neuron_input_batch\\"
# ds_train = tfds.load(
#     "mnist",
#     split="train",
#     as_supervised=True,    # yields (image, label) tuples
#     shuffle_files=False, 
# )
# ds_test = tfds.load(
#     "mnist",
#     split="test",
#     as_supervised=True,    # yields (image, label) tuples
#     shuffle_files=False,
# )
# import os
# import numpy as np
# ds_train_batched = ds_train.batch(batch_size)
# for noise in noiseAmount:
#     for batch_idx, (imgs, lbls) in enumerate(tfds.as_numpy(ds_train_batched)):
#         print(f"Batch {batch_idx}: {imgs.shape}, Labels: {lbls.shape}")
#         batch_name = f"batch_{batch_idx}--{batch_size}" + ".npy"
#         conv_labels_name = f"conv_labels_batch_{batch_idx}--50.npy"
#         imgs = imgs.squeeze(-1)
#         print(imgs.shape)
#         os.makedirs(conv_dir, exist_ok=True)
#         os.makedirs(neuron_input_dir, exist_ok=True)
#         convolved_batch_location = conv_dir + batch_name
#         neuron_input_image_batch_location = neuron_input_dir + batch_name
#         conv_labels_name = conv_dir + conv_labels_name
#         np.save(conv_labels_name, lbls)
#         # upscaled_images = upscale_mnist(imgs, target_size=128, method='bicubic')
#         # print(f"Original shape: {imgs.shape}, Upscaled shape: {upscaled_images.shape}")
#         # convolved_images = convolve_images(upscaled_images,
#         #                                    gabor_filters,
#         #                                    convolved_batch_location)

# Generate Data on Latency and 

In [4]:
import os
import numpy as np
import tensorflow_datasets as tfds
from PIL import Image
from scipy.signal import fftconvolve

# === CONFIGURATION ===
batch_size = 50
noise_levels = [0.0, 5.0, 15.0, 30.0, 50.0]  # std-dev of Gaussian noise
conv_dir = "/home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/conv_mnist_batch/"
neuron_input_dir = "/home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/"
filter_dir = "/home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/configs/input/filters"

gabor_filters = load_gabor_filters(filter_dir)

# === LOAD MNIST DATA ===
ds_train = tfds.load("mnist", split="train", as_supervised=True, shuffle_files=False)
ds_test = tfds.load("mnist", split="test", as_supervised=True, shuffle_files=False)

# === FUNCTIONS ===
def add_gaussian_noise_raw(imgs: np.ndarray, noise_std: float, clip: bool = True) -> np.ndarray:
    imgs = imgs.astype(np.float32)
    noise = np.random.normal(loc=0.0, scale=noise_std, size=imgs.shape)
    noisy_imgs = imgs + noise
    if clip:
        noisy_imgs = np.clip(noisy_imgs, 0.0, 255.0)
        return noisy_imgs.astype(np.uint8)
    return noisy_imgs

def upscale_mnist(images: np.ndarray, target_size: int = 128, method: str = 'bicubic') -> np.ndarray:
    """Upscale MNIST images to target resolution."""
    resample_method = Image.BICUBIC if method == 'bicubic' else Image.BILINEAR
    return np.stack([
        np.array(Image.fromarray(img).resize((target_size, target_size), resample=resample_method))
        for img in images
    ])

def convolve_images(images: np.ndarray, gabor_filters: list[np.ndarray], save_path: str, normalise: bool = True) -> np.ndarray:
    num_images, H, W = images.shape
    filters = np.stack(gabor_filters, axis=0)
    F, kH, kW = filters.shape
    convolved = np.zeros((num_images, F, H, W), dtype=np.result_type(images, filters))

    for i in range(num_images):
        img_stack = np.broadcast_to(images[i], (F, H, W))
        convolved[i] = fftconvolve(img_stack, filters, mode='same', axes=(1, 2))
        if normalise:
            norms = np.linalg.norm(convolved[i], axis=(1, 2), keepdims=True)
            convolved[i] /= norms

    np.save(save_path, convolved)
    return convolved

os.makedirs(conv_dir, exist_ok=True)
os.makedirs(neuron_input_dir, exist_ok=True)
mapping_path = "/home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/configs/input/mapping.npz"
# === MAIN PIPELINE ===
def save_mnist_with_noise_and_convolution_and_mapping(dataset, prefix: str):
    dataset = dataset.batch(batch_size)
    for noise in noise_levels:
        base_dir = os.path.join(neuron_input_dir, f"noise_{int(noise)}", prefix)
        os.makedirs(base_dir, exist_ok=True)

        for batch_idx, (imgs, lbls) in enumerate(tfds.as_numpy(dataset)):
            if batch_idx % 100 == 0:
                print(f"Processing batch {batch_idx} with noise level {noise}")
            imgs = np.squeeze(imgs, axis=-1)  # (B, 28, 28)
            noisy_imgs = add_gaussian_noise_raw(imgs, noise_std=noise)

            batch_name = os.path.join(base_dir, f"neuron_input_batch_{batch_idx}.npy")
            label_name = os.path.join(base_dir, f"labels_{batch_idx}.npy")

            conv_labels_location = os.path.join(base_dir, label_name)
            neuron_input_path = os.path.join(base_dir, batch_name)
            # Save labels
            np.save(conv_labels_location, lbls)

            # Upscale and convolve
            upscaled_images = upscale_mnist(noisy_imgs, target_size=128, method='bicubic')
            convolved_images = convolve_images_without_saving(upscaled_images, gabor_filters)
            generate_neuron_inputs_from_array(
                convolved_images,
                mapping_path,
                neuron_input_path
            )



# Run for train and test sets
save_mnist_with_noise_and_convolution_and_mapping(ds_train, prefix="train")
save_mnist_with_noise_and_convolution_and_mapping(ds_test, prefix="test")


Loaded filter from gabor_l0.8_b1_t2.36_p0.00_g0.5.npy, shape: (6, 6)
Loaded filter from gabor_l0.8_b1_t0.00_p0.00_g0.5.npy, shape: (6, 6)
Loaded filter from gabor_l0.8_b1_t0.79_p3.14_g0.5.npy, shape: (6, 6)
Loaded filter from gabor_l0.8_b1_t1.57_p0.00_g0.5.npy, shape: (6, 6)
Loaded filter from gabor_l0.8_b1_t2.36_p3.14_g0.5.npy, shape: (6, 6)
Loaded filter from gabor_l0.8_b1_t0.00_p3.14_g0.5.npy, shape: (6, 6)
Loaded filter from gabor_l0.8_b1_t0.79_p0.00_g0.5.npy, shape: (6, 6)
Loaded filter from gabor_l0.8_b1_t1.57_p3.14_g0.5.npy, shape: (6, 6)
Loaded 8 Gabor filters


I0000 00:00:1750431693.308380   16502 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7498 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:01:00.0, compute capability: 6.1
I0000 00:00:1750431693.309172   16502 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 7498 MB memory:  -> device: 1, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:05:00.0, compute capability: 6.1
I0000 00:00:1750431693.309833   16502 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 7498 MB memory:  -> device: 2, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:09:00.0, compute capability: 6.1
I0000 00:00:1750431693.310415   16502 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 7498 MB memory:  -> device: 3, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:84:00.0, compute capability: 6.1
I0000 00:00:1750431693.310993   16502 gpu_device.cc:2019] Cr

Processing batch 0 with noise level 0.0
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_0/train/neuron_input_batch_0.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_0/train/neuron_input_batch_1.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_0/train/neuron_input_batch_2.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_0/train/neuron_input_batch_3.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_0/train/neuron_input_batch_4.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_0/train/neuron_input_batch_5.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/m

2025-06-20 15:09:24.973750: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_5/train/neuron_input_batch_0.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_5/train/neuron_input_batch_1.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_5/train/neuron_input_batch_2.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_5/train/neuron_input_batch_3.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_5/train/neuron_input_batch_4.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_5/train/neuron_input_batch_5.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_i

...

Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_15/train/neuron_input_batch_0.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_15/train/neuron_input_batch_1.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_15/train/neuron_input_batch_2.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_15/train/neuron_input_batch_3.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_15/train/neuron_input_batch_4.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_15/train/neuron_input_batch_5.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/ne

2025-06-20 15:33:11.353727: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_50/train/neuron_input_batch_0.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_50/train/neuron_input_batch_1.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_50/train/neuron_input_batch_2.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_50/train/neuron_input_batch_3.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_50/train/neuron_input_batch_4.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_50/train/neuron_input_batch_5.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/ne

2025-06-20 15:45:11.835934: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_30/test/neuron_input_batch_0.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_30/test/neuron_input_batch_1.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_30/test/neuron_input_batch_2.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_30/test/neuron_input_batch_3.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_30/test/neuron_input_batch_4.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_input_batch/noise_30/test/neuron_input_batch_5.npy
Neuron inputs saved to /home/jake/Document/Spikes/projects/mnist_class/mnist_class_wip/data/neuron_i